# Text-Fabric to Neo4j

## Table of content (ToC)<a class="anchor" id="TOC"></a>
* <a href="#bullet1">1 - Introduction</a>
* <a href="#bullet2">2 - Setup</a>
* <a href="#bullet3">3 - Creating the Neo4j data</a>
* <a href="#bullet4">4 - Some tests</a>
* <a href="#bullet5">5 - Notebook version</a>

#  1 - Introduction <a class="anchor" id="bullet1"></a>
##### [Back to ToC](#TOC)

Use this notebook to convert a Text-Fabric dataset into a Neo4j graph.

- Nodes become `(:<tf_otype> {tf_id, otype, ...features})` (no shared TFNode label)
- Every TF edge-feature becomes its own Neo4j relationship type
- Optional: sequential TF links `[:NEXT]` and `[:PREVIOUS]`

Set paths and credentials in the config cell, then run all cells.

# 2 - Setup <a class="anchor" id="bullet2"></a>
##### [Back to ToC](#TOC)

Setup the environment.

In [1]:
%pip install text-fabric neo4j python-dotenv

Note: you may need to restart the kernel to use updated packages.


# 3 - Creating the Neo4j data <a class="anchor" id="bullet3"></a>
##### [Back to ToC](#TOC)

first set the variables.

In [1]:
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

load_dotenv()

# Make sure src/ is importable when notebook is started from repo root or notebooks/
repo_root = Path.cwd()
if not (repo_root / "src").exists() and (repo_root.parent / "src").exists():
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root / "src"))

from tf2neo4j import TFExportConfig, export_text_fabric_to_neo4j

In [2]:
# --- Configure your dataset + Neo4j connection ---
# Text-Fabric dataset folder (required)
TF_LOCATIONS = r"C:\Users\tonyj\text-fabric-data\github\CenterBLC\N1904\tf\1.0.0"
#TF_LOCATIONS = r"C:\Users\tonyj\text-fabric-data\github\codykingham\tischendorf_tf\tf\2.8"

# Optional TF module name/list. Keep as None for plain local data.
TF_MODULES = None

NEO4J_URI = os.getenv("NEO4J_URI", "bolt://localhost:7687")
NEO4J_USER = os.getenv("NEO4J_USER", "neo4j")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD", "neo4j")
NEO4J_DATABASE = os.getenv("NEO4J_DATABASE", "neo4j")

# None means: use all available TF node/edge features
NODE_FEATURES = None
EDGE_FEATURES = None

# Tuning
BATCH_SIZE = 2000
# Use True on first run when switching from the old TF_EDGE schema
CLEAR_DATABASE = True
# Delete chunk size when CLEAR_DATABASE=True (lower if memory is tight)
CLEAR_BATCH_SIZE = 2000
ADD_PREVIOUS_NEXT = True
# When enabled, only these node types get NEXT/PREVIOUS links. None => all types.
PREVIOUS_NEXT_NODE_TYPES = ["book", "chapter","verse", "sentence", "word"]
# Locality-based hierarchy list (adjacent pairs are linked as TF_HIERARCHY)
HIERARCHY_NODE_TYPES = ["book", "chapter", "verse", "word"]
# Convert frame role codes to semantic relations (A0/A1/A2/AA2)
FRAME_SEMANTIC_RELATIONS = True
SHOW_PROGRESS = True
PROGRESS_USE_TQDM = True
PROGRESS_EVERY = 5000

Now let us check the connection (small helper function).

In [3]:
from neo4j import GraphDatabase
with GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD)) as d:
    d.verify_connectivity()
print("Neo4j connection OK")

Neo4j connection OK


Now the real converion is done by calling the python code.

In [4]:
config = TFExportConfig(
    tf_locations=TF_LOCATIONS,
    tf_modules=TF_MODULES,
    neo4j_uri=NEO4J_URI,
    neo4j_user=NEO4J_USER,
    neo4j_password=NEO4J_PASSWORD,
    neo4j_database=NEO4J_DATABASE,
    node_features=NODE_FEATURES,
    edge_features=EDGE_FEATURES,
    batch_size=BATCH_SIZE,
    clear_database=CLEAR_DATABASE,
    clear_batch_size=CLEAR_BATCH_SIZE,
    add_previous_next=ADD_PREVIOUS_NEXT,
    previous_next_node_types=PREVIOUS_NEXT_NODE_TYPES,
    hierarchy_node_types=HIERARCHY_NODE_TYPES,
    frame_semantic_relations=FRAME_SEMANTIC_RELATIONS,
    show_progress=SHOW_PROGRESS,
    progress_use_tqdm=PROGRESS_USE_TQDM,
    progress_every=PROGRESS_EVERY,
)

stats = export_text_fabric_to_neo4j(config)
print(f"Export complete: {stats.node_count} nodes, {stats.relationship_count} relationships")

[tf2neo4j] Loading Text-Fabric API
  1.62s Feature overview: 58 for nodes; 5 for edges; 1 configs; 9 computed
[tf2neo4j] Prepared export: 497525 nodes, 5 edge features
[tf2neo4j] Connecting to Neo4j
[tf2neo4j] Clearing existing database
[tf2neo4j] Ensuring per-label tf_id constraints
[tf2neo4j] Constraint ready for :book(tf_id)
[tf2neo4j] Constraint ready for :chapter(tf_id)
[tf2neo4j] Constraint ready for :clause(tf_id)
[tf2neo4j] Constraint ready for :group(tf_id)
[tf2neo4j] Constraint ready for :phrase(tf_id)
[tf2neo4j] Constraint ready for :sentence(tf_id)
[tf2neo4j] Constraint ready for :subphrase(tf_id)
[tf2neo4j] Constraint ready for :verse(tf_id)
[tf2neo4j] Constraint ready for :wg(tf_id)
[tf2neo4j] Constraint ready for :word(tf_id)


Writing nodes:   0%|          | 0/497525 [00:00<?, ?nodes/s]

[tf2neo4j] Refreshing semantic frame relations
[tf2neo4j] Writing semantic frame relations


Edge frame (semantic): 0rels [00:00, ?rels/s]

[tf2neo4j] Writing edge feature 'oslots'


...

[tf2neo4j] Hierarchy book -> chapter
[tf2neo4j] Hierarchy chapter -> verse
[tf2neo4j] Hierarchy verse -> word
[tf2neo4j] Refreshing NEXT/PREVIOUS relationships
[tf2neo4j] Writing NEXT relationships


Edge NEXT:   0%|          | 0/154016 [00:00<?, ?rels/s]

[tf2neo4j] Writing PREVIOUS relationships


Edge PREVIOUS:   0%|          | 0/154016 [00:00<?, ?rels/s]

[tf2neo4j] Export complete: 497525 nodes, 7229525 relationships
Export complete: 497525 nodes, 7229525 relationships


# 4 - Some tests <a class="anchor" id="bullet4"></a>
##### [Back to ToC](#TOC)

In [5]:
from neo4j import GraphDatabase

with GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD)) as driver:
    with driver.session(database=NEO4J_DATABASE) as session:
        n = session.run("MATCH (n) WHERE 'TFNode' IN labels(n) RETURN count(n) AS c").single()["c"]
        r = session.run("MATCH ()-[r]->() RETURN count(r) AS c").single()["c"]
        rel_types = session.run("MATCH ()-[r]->() RETURN type(r) AS t, count(*) AS c ORDER BY c DESC LIMIT 15").data()
        labels = session.run("MATCH (n) WHERE n.tf_id IS NOT NULL UNWIND labels(n) AS l RETURN l, count(*) AS c ORDER BY c DESC").data()
        tfnode_left = session.run("MATCH (n) WHERE 'TFNode' IN labels(n) RETURN count(n) AS c").single()["c"]

print({"nodes": n, "relationships": r})
print("Top relationship types:", rel_types)
print("Node labels:", labels)
print("Remaining TFNode labels:", tfnode_left)

{'nodes': 0, 'relationships': 7229525}
Top relationship types: [{'t': 'sibling', 'c': 6200411}, {'t': 'parent', 'c': 510894}, {'t': 'NEXT', 'c': 154016}, {'t': 'PREVIOUS', 'c': 154016}, {'t': 'TF_HIERARCHY', 'c': 145983}, {'t': 'frame', 'c': 43893}, {'t': 'subjref', 'c': 20312}]
Node labels: [{'l': 'word', 'c': 137779}, {'l': 'subphrase', 'c': 116178}, {'l': 'wg', 'c': 106868}, {'l': 'phrase', 'c': 69007}, {'l': 'clause', 'c': 42506}, {'l': 'group', 'c': 8945}, {'l': 'sentence', 'c': 8011}, {'l': 'verse', 'c': 7944}, {'l': 'chapter', 'c': 260}, {'l': 'book', 'c': 27}]
Remaining TFNode labels: 0


Quick checks that can be done in Neo4j desktop:

<pre>
MATCH (n) WHERE n.tTFNode IS NOT NULL UNWIND labels(n) AS l
RETURN l, count(*) AS c
ORDER BY c DESC;
</pre>

<pre>
MATCH ()-[r]->()
RETURN type(r) AS rel_type, count(*) AS c
ORDER BY c DESC
LIMIT 25;
</pre>

<pre>
MATCH (a)-[:NEXT]->(b)
WHERE a.tf_id IS NOT NULL AND b.tf_id IS NOT NULL
RETURN a.tf_id, b.tf_id
LIMIT 10;
</pre>

# 5 - Notebook version<a class="anchor" id="bullet5"></a>
##### [Back to ToC](#TOC)

<div style="float: left;">
  <table>
    <tr>
      <td><strong>Author</strong></td>
      <td>Tony Jurg</td>
    </tr>
    <tr>
      <td><strong>Version</strong></td>
      <td>1.0</td>
    </tr>
    <tr>
      <td><strong>Date</strong></td>
      <td>March 9, 2026</td>
    </tr>
  </table>
</div>